# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Ravikiranbathe/flyrank-ai/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

The goal is to rank pages that may need a content refresh and explain why they were selected. I use the Week-5 baseline score to prioritize pages and assign simple reason codes based on the measured signals.

In [6]:
import os
import pandas as pd

# Clone the repository if it is not already available
if not os.path.exists("/content/flyrank-ai"):
    !git clone https://github.com/Ravikiranbathe/flyrank-ai.git

# Move into the repository
os.chdir("/content/flyrank-ai")

# Load dataset
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

print("Current folder:", os.getcwd())
print("Dataset shape:", df.shape)

Current folder: /content/flyrank-ai
Dataset shape: (30000, 44)


In [7]:
# Create the Week-5 baseline score

df["baseline_score"] = 0

# Add points for each refresh signal
df.loc[df["content_age_days"] >= 365, "baseline_score"] += 40
df.loc[df["trend_pct"] < -10, "baseline_score"] += 25
df.loc[df["ctr"] < 2, "baseline_score"] += 20
df.loc[df["impressions_90d"] >= 1000, "baseline_score"] += 15

# Create recommended action
df["action"] = df["baseline_score"].apply(
    lambda x: "Refresh Content" if x >= 60 else "Monitor"
)

# Create reason codes
def get_reason_codes(row):
    reasons = []

    if row["content_age_days"] >= 365:
        reasons.append("STALE")

    if row["trend_pct"] < -10:
        reasons.append("DECLINING")

    if row["ctr"] < 2:
        reasons.append("LOW_CTR")

    if row["impressions_90d"] >= 1000:
        reasons.append("VISIBLE")

    if not reasons:
        reasons.append("LOW_PRIORITY")

    return ", ".join(reasons)

df["reason_codes"] = df.apply(get_reason_codes, axis=1)

# Rank pages from highest score to lowest
ranked_queue = df.sort_values(
    ["baseline_score", "impressions_90d"],
    ascending=[False, False]
).copy()

ranked_queue["rank"] = range(1, len(ranked_queue) + 1)

# Show top 20 pages
ranked_queue[
    [
        "rank",
        "baseline_score",
        "action",
        "reason_codes",
        "content_age_days",
        "trend_pct",
        "ctr",
        "impressions_90d"
    ]
].head(20)

,rank,baseline_score,action,reason_codes,content_age_days,trend_pct,ctr,impressions_90d
6653,1,100,Refresh Content,"STALE, DECLINING, LOW_CTR, VISIBLE",537,-44.8,0.14,517715
26844,2,100,Refresh Content,"STALE, DECLINING, LOW_CTR, VISIBLE",445,-44.5,0.15,509252
21819,3,100,Refresh Content,"STALE, DECLINING, LOW_CTR, VISIBLE",445,-33.2,0.41,463103
29879,4,100,Refresh Content,"STALE, DECLINING, LOW_CTR, VISIBLE",482,-27.0,0.23,416180
21565,5,100,Refresh Content,"STALE, DECLINING, LOW_CTR, VISIBLE",445,-37.3,0.87,309192
22402,6,100,Refresh Content,"STALE, DECLINING, LOW_CTR, VISIBLE",480,-25.0,0.29,192478
29998,7,100,Refresh Content,"STALE, DECLINING, LOW_CTR, VISIBLE",466,-27.9,0.22,154763
16736,8,100,Refresh Content,"STALE, DECLINING, LOW_CTR, VISIBLE",445,-14.8,0.07,149712
15998,9,100,Refresh Content,"STALE, DECLINING, LOW_CTR, VISIBLE",480,-15.1,0.54,144195
3232,10,100,Refresh Content,"STALE, DECLINING, LOW_CTR, VISIBLE",480,-14.3,0.44,140776


### Ranked Queue Interpretation

The highest-ranked pages have the strongest combination of refresh signals. The reason codes explain why each page was selected.

The ranking is used to prioritize human review rather than automatically deciding what changes should be made.

## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

This score is intended to support human review and prioritize pages for possible content refresh.

It should not be treated as a prediction of guaranteed traffic, ranking improvement, or business impact. The score is based only on the signals available in this dataset.

The final decision to refresh a page should be made after reviewing the actual content and its context.

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

Every recommended page should be reviewed by a human before making changes.

### Human Review

The reviewer should check:
- Whether the content is actually outdated
- Whether the declining trend has a clear reason
- Whether the page is still relevant
- Whether the search intent has changed
- Whether the recommended refresh is worth the effort

### No-Go List

The score should NOT automatically:
- Publish content
- Delete pages
- Create redirects
- Merge pages
- Rewrite important content
- Make high-stakes claims
- Guarantee ranking or traffic improvement

The score is a prioritization tool, not an automatic decision maker.

In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

I would review the scoring rules if the input data changes significantly, the percentage of pages marked for refresh changes, or one signal starts dominating the ranked queue.

I would also review the approach if human reviewers frequently disagree with the recommendations or if observed outcomes do not match the recommendations.

A new independently observed outcome would be especially useful for future validation because the current target is based on the same signals used by the model.

In [10]:
# Assign an operational archetype to each page

def assign_archetype(row):
    age = row["content_age_days"] >= 365
    decline = row["trend_pct"] < -10
    low_ctr = row["ctr"] < 2
    visible = row["impressions_90d"] >= 1000

    if age and visible:
        return "Stale + visible"
    elif decline and visible:
        return "Declining + visible"
    elif low_ctr and visible:
        return "Low CTR + visible"
    elif sum([age, decline, low_ctr, visible]) >= 3:
        return "Multi-signal opportunity"
    else:
        return "Low-priority / stable"


ranked_queue["archetype"] = ranked_queue.apply(
    assign_archetype, axis=1
)

# Map each archetype to an action
action_map = {
    "Stale + visible": "Refresh content",
    "Declining + visible": "Investigate and refresh",
    "Low CTR + visible": "Review title/meta",
    "Multi-signal opportunity": "High-priority review",
    "Low-priority / stable": "Monitor"
}

ranked_queue["suggested_action"] = ranked_queue["archetype"].map(action_map)

# Display top 20
ranked_queue[
    ["rank", "baseline_score", "archetype", "suggested_action"]
].head(20)

,rank,baseline_score,archetype,suggested_action
6653,1,100,Stale + visible,Refresh content
26844,2,100,Stale + visible,Refresh content
21819,3,100,Stale + visible,Refresh content
29879,4,100,Stale + visible,Refresh content
21565,5,100,Stale + visible,Refresh content
22402,6,100,Stale + visible,Refresh content
29998,7,100,Stale + visible,Refresh content
16736,8,100,Stale + visible,Refresh content
15998,9,100,Stale + visible,Refresh content
3232,10,100,Stale + visible,Refresh content


### Cost / Value Thinking

The ranked queue can help focus limited content-review time on pages with stronger refresh signals.

A practical approach is to review the highest-ranked pages first and compare the expected effort of a refresh with the potential value of the page, such as its existing visibility and traffic.

This is a prioritization idea, not a measured business-value result.

### Decay / Refresh Insight

Content age and declining performance are useful signals for identifying pages that may need attention.

However, age alone does not mean that a page should be refreshed. The signals should be considered together with human review and the actual content context.


## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

The ranked action queue is exported so it can be reused in the paper and reviewed separately.

The export contains the page ranking, baseline score, recommended action, reason codes, archetype, and suggested action.

In [11]:
# Export the ranked action queue

import os

os.makedirs("work/outputs", exist_ok=True)

output_path = "work/outputs/w07_ranked_action_queue.csv"

ranked_queue.to_csv(output_path, index=False)

print("Saved:", output_path)
print("Rows exported:", len(ranked_queue))

Saved: work/outputs/w07_ranked_action_queue.csv
Rows exported: 30000


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled with the required markdown thinking and supporting code where needed
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.